# 📚 Liane's Library — CRUD Operations

Before writing any sort of application, CRUD functions need to be developed and tested. This notebook provides a space for that.

> **C**reate · **R**ead · **U**pdate · **D**elete · **V**alidate

---

## 0. Setup — imports and connection 🔌

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from datetime import date

In [2]:
load_dotenv(override=True)

schema   = "lianes_library"
host     = "127.0.0.1"
user     = "root"
password = os.getenv("MYSQL_PASSWORD")
port     = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)

print("Connected to lianes_library!")

Connected to lianes_library!


---
# CREATE ➕
Functions to add new records into the database.

## Define

In [3]:
def create_book(isbn, book_name, author, genre=None,
                published_year=None, total_pages=None,
                book_condition='good', mood_tags=None):
    """
    Adds a new book to Liane's collection.
    book_condition: mint, good, worn, damaged
    mood_tags example: 'cosy,dark,inspiring'
    """
    df = pd.DataFrame([{
        "isbn"          : isbn,
        "book_name"     : book_name,
        "author"        : author,
        "genre"         : genre,
        "published_year": published_year,
        "total_pages"   : total_pages,
        "book_condition": book_condition,
        "mood_tags"     : mood_tags,
        "is_available"  : True
    }])
    df.to_sql("books", if_exists="append", con=connection_string, index=False)
    return f"📖 '{book_name}' by {author} added to the library!"

In [4]:
def create_friend(friend_name, phone_number=None, email=None,
                  max_loans=3, trust_score=100,
                  preferred_genres=None, notes=None):
    """
    Adds a new friend/borrower to the system.
    trust_score starts at 100 and goes down for bad behaviour.
    """
    df = pd.DataFrame([{
        "friend_name"     : friend_name,
        "phone_number"    : phone_number,
        "email"           : email,
        "max_loans"       : max_loans,
        "trust_score"     : trust_score,
        "preferred_genres": preferred_genres,
        "notes"           : notes
    }])
    df.to_sql("friends", if_exists="append", con=connection_string, index=False)
    return f"👤 '{friend_name}' added as a borrower!"

In [5]:
def create_loan(friend, book, loan_date=None, due_days=14, notes=None):
    """
    Records a new loan.
    due_days = how many days until the book is due back (default 14).
    renewal_date is set automatically to due_date + 7 days.
    tracker_id is generated automatically by the database trigger.
    """
    if loan_date is None:
        loan_date = date.today()

    due_date     = pd.Timestamp(loan_date) + pd.Timedelta(due_days, "D")
    renewal_date = due_date + pd.Timedelta(7, "D")

    df = pd.DataFrame([{
        "isbn"        : book["isbn"],
        "friend_id"   : friend["friend_id"],
        "loan_date"   : loan_date,
        "due_date"    : due_date.date(),
        "renewal_date": renewal_date.date(),
        "notes"       : notes
    }])
    df.to_sql("loans", if_exists="append", con=connection_string, index=False)
    return f"📋 '{friend['friend_name']}' borrowed '{book['book_name']}'. Due: {due_date.date()}"

In [6]:
def create_reading_session(loan_id, pages_read, mood_rating=None,
                            session_notes=None, session_date=None):
    """
    Logs a reading session for a loan.
    mood_rating: 1 (terrible) to 5 (amazing)
    """
    if session_date is None:
        session_date = date.today()

    df = pd.DataFrame([{
        "loan_id"      : loan_id,
        "session_date" : session_date,
        "pages_read"   : pages_read,
        "mood_rating"  : mood_rating,
        "session_notes": session_notes
    }])
    df.to_sql("reading_sessions", if_exists="append", con=connection_string, index=False)
    return f"📖 Reading session logged! {pages_read} pages read for loan {loan_id}."

In [7]:
def create_wishlist_request(friend, book_title, author=None):
    """
    Adds a book request to the wishlist.
    Used when a friend wants a book Liane does not own yet.
    """
    df = pd.DataFrame([{
        "friend_id" : friend["friend_id"],
        "book_title": book_title,
        "author"    : author,
        "fulfilled" : False
    }])
    df.to_sql("wishlist", if_exists="append", con=connection_string, index=False)
    return f"🎁 '{friend['friend_name']}' requested '{book_title}'. Added to wishlist!"

In [8]:
def create_review(loan_id, rating, review_text=None):
    """
    Adds a review after a book is returned.
    rating: 1 to 5 stars.
    Only one review allowed per loan.
    """
    df = pd.DataFrame([{
        "loan_id"    : loan_id,
        "rating"     : rating,
        "review_text": review_text,
        "reviewed_on": date.today()
    }])
    df.to_sql("reviews", if_exists="append", con=connection_string, index=False)
    stars = "⭐" * rating
    return f"{stars} Review submitted for loan {loan_id}!"

## Test CREATE

In [9]:
# Add 3 brand new books
print(create_book("9780307949486", "The Wind-Up Bird Chronicle", "Haruki Murakami", "Fiction",  1994, 607,  "mint", "mysterious,cosy"))
print(create_book("9780156012195", "The Little Prince",          "Antoine de Saint-Exupery", "Fiction", 1943, 96, "good", "cosy,inspiring"))
print(create_book("9780385333481", "Watership Down",             "Richard Adams",   "Fiction",  1972, 413, "worn", "emotional,inspiring"))

pd.read_sql("SELECT isbn, book_name, author, genre FROM books", con=connection_string)

📖 'The Wind-Up Bird Chronicle' by Haruki Murakami added to the library!
📖 'The Little Prince' by Antoine de Saint-Exupery added to the library!
📖 'Watership Down' by Richard Adams added to the library!


,isbn,book_name,author,genre
0,9780062316097,Sapiens,Yuval Noah Harari,Non-Fiction
1,9780141036144,To Kill a Mockingbird,Harper Lee,Fiction
2,9780143127741,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction
3,9780156012195,The Little Prince,Antoine de Saint-Exupery,Fiction
4,9780241984536,The Alchemist,Paulo Coelho,Fiction
5,9780307949486,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction
6,9780316769174,The Catcher in the Rye,J.D. Salinger,Fiction
7,9780385333481,Watership Down,Richard Adams,Fiction
8,9780385490818,The Handmaids Tale,Margaret Atwood,Dystopian
9,9780525559474,The Subtle Art of Not Giving a F,Mark Manson,Self-Help


In [10]:
# Add 3 brand new friends
print(create_friend("Priya Sharma",  "07700111222", "priya.sharma@email.com",  3, 100, "Fiction,Romance",   "Very reliable"))
print(create_friend("Carlos Rivera", "07700333444", "carlos.rivera@email.com", 2, 90,  "Fiction,Classic",   "Returns on time"))
print(create_friend("Mei Lin",       "07700555666", "mei.lin@email.com",       3, 100, "Self-Help,Fiction", "Always careful with books"))

pd.read_sql("SELECT friend_id, friend_name, email, trust_score FROM friends", con=connection_string)

👤 'Priya Sharma' added as a borrower!
👤 'Carlos Rivera' added as a borrower!
👤 'Mei Lin' added as a borrower!


,friend_id,friend_name,email,trust_score
0,1,Emma Watson,emma@email.com,100
1,2,James Brown,james@email.com,55
2,3,Sophie Turner,sophie@email.com,90
3,4,Liam Smith,liam@email.com,60
4,5,Olivia Jones,olivia@email.com,100
5,6,Liane herself,liane@library.com,100
6,16,Ashritha,ashritha@email.com,100
7,17,Noah Bennett,noah@email.com,90
8,18,Zara Ahmed,zara@email.com,100
9,19,Priya Sharma,priya.sharma@email.com,100


In [11]:
# Fetch the newly added friends and books then create loans
priya   = pd.read_sql("SELECT * FROM friends WHERE email = 'priya.sharma@email.com'",  con=connection_string).iloc[0]
carlos  = pd.read_sql("SELECT * FROM friends WHERE email = 'carlos.rivera@email.com'", con=connection_string).iloc[0]
mei     = pd.read_sql("SELECT * FROM friends WHERE email = 'mei.lin@email.com'",       con=connection_string).iloc[0]

book1   = pd.read_sql("SELECT * FROM books WHERE isbn = '9780307949486'", con=connection_string).iloc[0]
book2   = pd.read_sql("SELECT * FROM books WHERE isbn = '9780156012195'", con=connection_string).iloc[0]
book3   = pd.read_sql("SELECT * FROM books WHERE isbn = '9780385333481'", con=connection_string).iloc[0]

print(create_loan(priya,  book1, due_days=14, notes="Loves Murakami"))
print(create_loan(carlos, book2, due_days=21, notes="Reading with his daughter"))
print(create_loan(mei,    book3, due_days=14, notes="For her book club"))

pd.read_sql("SELECT loan_id, tracker_id, isbn, friend_id, loan_date, due_date FROM loans ORDER BY loan_id DESC LIMIT 5", con=connection_string)

📋 'Priya Sharma' borrowed 'The Wind-Up Bird Chronicle'. Due: 2026-09-28
📋 'Carlos Rivera' borrowed 'The Little Prince'. Due: 2026-10-05
📋 'Mei Lin' borrowed 'Watership Down'. Due: 2026-09-28


,loan_id,tracker_id,isbn,friend_id,loan_date,due_date
0,18,LIB-000018,9780385333481,21,2026-09-14,2026-09-28
1,17,LIB-000017,9780156012195,20,2026-09-14,2026-10-05
2,16,LIB-000016,9780307949486,19,2026-09-14,2026-09-28
3,15,LIB-000015,9781501173219,18,2026-09-14,2026-09-28
4,14,LIB-000012,9780143127741,17,2026-09-14,2026-10-05


In [12]:
# Log reading sessions for the 3 new loans
new_loans = pd.read_sql("SELECT * FROM loans ORDER BY loan_id DESC LIMIT 3", con=connection_string)

print(create_reading_session(new_loans.iloc[0]["loan_id"], pages_read=120, mood_rating=5, session_notes="Beautifully written!"))
print(create_reading_session(new_loans.iloc[1]["loan_id"], pages_read=50,  mood_rating=5, session_notes="His daughter loved it too"))
print(create_reading_session(new_loans.iloc[2]["loan_id"], pages_read=80,  mood_rating=4, session_notes="Great discussion at book club"))

pd.read_sql("SELECT * FROM reading_sessions ORDER BY session_id DESC LIMIT 5", con=connection_string)

📖 Reading session logged! 120 pages read for loan 18.
📖 Reading session logged! 50 pages read for loan 17.
📖 Reading session logged! 80 pages read for loan 16.


,session_id,loan_id,session_date,pages_read,mood_rating,session_notes
0,17,16,2026-09-14,80,4,Great discussion at book club
1,16,17,2026-09-14,50,5,His daughter loved it too
2,15,18,2026-09-14,120,5,Beautifully written!
3,14,11,2026-09-14,100,5,Could not put it down!
4,13,14,2026-09-14,55,4,Heavy themes but brilliant


In [13]:
# Add wishlist requests from new friends
print(create_wishlist_request(priya,  "Norwegian Wood",       "Haruki Murakami"))
print(create_wishlist_request(carlos, "One Hundred Years of Solitude", "Gabriel Garcia Marquez"))
print(create_wishlist_request(mei,    "Educated",             "Tara Westover"))

pd.read_sql("wishlist", con=connection_string)

🎁 'Priya Sharma' requested 'Norwegian Wood'. Added to wishlist!
🎁 'Carlos Rivera' requested 'One Hundred Years of Solitude'. Added to wishlist!
🎁 'Mei Lin' requested 'Educated'. Added to wishlist!


,wishlist_id,friend_id,book_title,author,requested_on,fulfilled
0,1,1,The Alchemist,Paulo Coelho,2024-08-10,1
1,2,2,Thinking Fast and Slow,Daniel Kahneman,2024-08-12,0
2,3,3,Pride and Prejudice,Jane Austen,2024-08-15,0
3,4,4,1984,George Orwell,2024-08-18,0
4,5,5,The Power of Now,Eckhart Tolle,2024-08-20,0
5,6,18,The Power of Now,Eckhart Tolle,2026-09-14,0
6,7,17,1984,George Orwell,2026-09-14,0
7,8,19,Norwegian Wood,Haruki Murakami,2026-09-14,0
8,9,20,One Hundred Years of Solitude,Gabriel Garcia Marquez,2026-09-14,0
9,10,21,Educated,Tara Westover,2026-09-14,0


---
# READ 🔍
Functions to retrieve and display data beautifully.

## Define

In [14]:
def prettify_df(df):
    """Makes column names clean and readable for display."""
    df = df.copy()
    df.columns = [
        c.upper() if c == "isbn" else c.replace("_", " ").capitalize()
        for c in df.columns
    ]
    return df.fillna("")

In [15]:
def read_books(available_only=False, genre=None, mood=None):
    """
    Reads all books.
    available_only=True  → only books on the shelf.
    genre='Fiction'      → filter by genre.
    mood='cosy'          → filter by mood tag.
    """
    books = pd.read_sql("books", con=connection_string)
    if available_only:
        books = books[books["is_available"] == 1]
    if genre:
        books = books[books["genre"].str.lower() == genre.lower()]
    if mood:
        books = books[books["mood_tags"].str.contains(mood, na=False)]
    return books

def display_books(books):
    """Returns a clean sorted view of books."""
    return books.pipe(prettify_df).sort_values(by="Book name")

In [16]:
def read_friends(min_trust=None):
    """
    Reads all friends.
    min_trust=80 → only friends with trust score 80 or above.
    """
    friends = pd.read_sql("friends", con=connection_string)
    if min_trust:
        friends = friends[friends["trust_score"] >= min_trust]
    return friends

def display_friends(friends):
    """Returns a clean sorted view of friends by trust score."""
    return friends.pipe(prettify_df).sort_values(by="Trust score", ascending=False)

In [17]:
def read_loans(active_only=False):
    """Reads all loans. active_only=True → only unreturned loans."""
    loans = pd.read_sql("loans", con=connection_string)
    if active_only:
        loans = loans[loans["return_date"].isna()]
    return loans

def display_loans():
    """Returns a joined view of loans with book and friend names."""
    return pd.read_sql("""
        SELECT
            l.tracker_id,
            f.friend_name,
            b.book_name,
            l.loan_date,
            l.due_date,
            l.renewal_date,
            CASE
                WHEN l.return_date IS NOT NULL                         THEN 'Returned'
                WHEN CURDATE() > COALESCE(l.renewal_date, l.due_date) THEN 'OVERDUE'
                WHEN l.renewal_date IS NOT NULL                       THEN 'Active - Renewed'
                ELSE 'Active'
            END AS status,
            l.fine_amount,
            l.fine_status
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        ORDER BY l.due_date ASC;
    """, con=connection_string)

In [18]:
def check_tracker(tracker_id):
    """Look up a single loan using its tracker ID like LIB-000001."""
    return pd.read_sql(f"""
        SELECT
            l.tracker_id,
            f.friend_name,
            b.book_name,
            l.loan_date,
            l.due_date,
            l.renewal_date,
            l.return_date,
            l.fine_amount,
            l.fine_status
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        WHERE l.tracker_id = '{tracker_id}';
    """, con=connection_string)

In [19]:
def read_overdue():
    """Returns all overdue loans with days overdue and fine calculated."""
    return pd.read_sql("""
        SELECT
            l.tracker_id,
            f.friend_name,
            f.phone_number,
            b.book_name,
            COALESCE(l.renewal_date, l.due_date)                            AS deadline,
            DATEDIFF(CURDATE(), COALESCE(l.renewal_date, l.due_date))       AS days_overdue,
            DATEDIFF(CURDATE(), COALESCE(l.renewal_date, l.due_date)) * 0.50 AS fine_due
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        WHERE l.return_date IS NULL
        AND CURDATE() > COALESCE(l.renewal_date, l.due_date)
        ORDER BY days_overdue DESC;
    """, con=connection_string)

In [20]:
def read_reading_progress():
    """Shows reading progress per loan as a percentage."""
    return pd.read_sql("""
        SELECT
            f.friend_name,
            b.book_name,
            b.total_pages,
            SUM(rs.pages_read)                                        AS pages_read_so_far,
            ROUND(SUM(rs.pages_read) * 100.0 / b.total_pages, 1)    AS percent_complete,
            ROUND(AVG(rs.mood_rating), 1)                            AS avg_mood
        FROM reading_sessions rs
        JOIN loans   l ON rs.loan_id  = l.loan_id
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        GROUP BY f.friend_name, b.book_name, b.total_pages
        ORDER BY percent_complete DESC;
    """, con=connection_string)

In [21]:
def library_summary():
    """One-line dashboard of the whole library."""
    return pd.read_sql("""
        SELECT
            (SELECT COUNT(*) FROM books)                              AS total_books,
            (SELECT COUNT(*) FROM books WHERE is_available = 1)      AS available,
            (SELECT COUNT(*) FROM friends)                           AS total_friends,
            (SELECT COUNT(*) FROM loans WHERE return_date IS NULL)   AS active_loans,
            (SELECT COUNT(*) FROM loans
             WHERE return_date IS NULL
             AND CURDATE() > COALESCE(renewal_date, due_date))       AS overdue_loans,
            (SELECT COALESCE(SUM(fine_amount), 0)
             FROM loans WHERE fine_status = 'unpaid')                AS unpaid_fines,
            (SELECT COUNT(*) FROM wishlist WHERE fulfilled = 0)      AS wishlist_pending;
    """, con=connection_string)

## Test READ

In [22]:
display_books(read_books())

,ISBN,Book name,Author,Genre,Published year,Total pages,Book condition,Mood tags,Cover url,Is available,Date added
10,9780593311295,Atomic Habits,James Clear,Self-Help,2018,320,good,"inspiring,educational",,1,2026-09-14
11,9780679720201,Crime and Punishment,Fyodor Dostoevsky,Classic,1901,545,damaged,"dark,intense",,1,2026-09-14
13,9780747532743,Harry Potter and the Philosophers Stone,J.K. Rowling,Fantasy,1997,223,good,"magical,cosy",,1,2026-09-14
16,9781501173219,It,Stephen King,Horror,1986,1138,good,"dark,intense",,1,2026-09-14
15,9781501156700,It Ends with Us,Colleen Hoover,Romance,2016,384,mint,"emotional,romantic",,1,2026-09-14
0,9780062316097,Sapiens,Yuval Noah Harari,Non-Fiction,2011,443,mint,"educational,inspiring",,1,2026-09-14
4,9780241984536,The Alchemist,Paulo Coelho,Fiction,1988,208,mint,"inspiring,magical",,1,2026-09-14
6,9780316769174,The Catcher in the Rye,J.D. Salinger,Fiction,1951,277,good,"dark,emotional",,1,2026-09-14
12,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Fiction,1925,180,worn,"dark,romantic",,1,2026-09-14
8,9780385490818,The Handmaids Tale,Margaret Atwood,Dystopian,1985,311,worn,"dark,emotional",,1,2026-09-14


In [23]:
display_books(read_books(available_only=True))

,ISBN,Book name,Author,Genre,Published year,Total pages,Book condition,Mood tags,Cover url,Is available,Date added
10,9780593311295,Atomic Habits,James Clear,Self-Help,2018,320,good,"inspiring,educational",,1,2026-09-14
11,9780679720201,Crime and Punishment,Fyodor Dostoevsky,Classic,1901,545,damaged,"dark,intense",,1,2026-09-14
13,9780747532743,Harry Potter and the Philosophers Stone,J.K. Rowling,Fantasy,1997,223,good,"magical,cosy",,1,2026-09-14
16,9781501173219,It,Stephen King,Horror,1986,1138,good,"dark,intense",,1,2026-09-14
15,9781501156700,It Ends with Us,Colleen Hoover,Romance,2016,384,mint,"emotional,romantic",,1,2026-09-14
0,9780062316097,Sapiens,Yuval Noah Harari,Non-Fiction,2011,443,mint,"educational,inspiring",,1,2026-09-14
4,9780241984536,The Alchemist,Paulo Coelho,Fiction,1988,208,mint,"inspiring,magical",,1,2026-09-14
6,9780316769174,The Catcher in the Rye,J.D. Salinger,Fiction,1951,277,good,"dark,emotional",,1,2026-09-14
12,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Fiction,1925,180,worn,"dark,romantic",,1,2026-09-14
8,9780385490818,The Handmaids Tale,Margaret Atwood,Dystopian,1985,311,worn,"dark,emotional",,1,2026-09-14


In [24]:
display_books(read_books(genre="Fiction"))

,ISBN,Book name,Author,Genre,Published year,Total pages,Book condition,Mood tags,Cover url,Is available,Date added
4,9780241984536,The Alchemist,Paulo Coelho,Fiction,1988,208,mint,"inspiring,magical",,1,2026-09-14
6,9780316769174,The Catcher in the Rye,J.D. Salinger,Fiction,1951,277,good,"dark,emotional",,1,2026-09-14
12,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Fiction,1925,180,worn,"dark,romantic",,1,2026-09-14
3,9780156012195,The Little Prince,Antoine de Saint-Exupery,Fiction,1943,96,good,"cosy,inspiring",,1,2026-09-14
14,9781250301697,The Midnight Library,Matt Haig,Fiction,2020,288,mint,"emotional,cosy",,1,2026-09-14
2,9780143127741,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,1994,607,mint,"mysterious,cosy",,1,2026-09-14
5,9780307949486,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,1994,607,mint,"mysterious,cosy",,1,2026-09-14
1,9780141036144,To Kill a Mockingbird,Harper Lee,Fiction,1960,281,good,"emotional,inspiring",,1,2026-09-14
7,9780385333481,Watership Down,Richard Adams,Fiction,1972,413,worn,"emotional,inspiring",,1,2026-09-14


In [25]:
display_books(read_books(mood="cosy"))

,ISBN,Book name,Author,Genre,Published year,Total pages,Book condition,Mood tags,Cover url,Is available,Date added
13,9780747532743,Harry Potter and the Philosophers Stone,J.K. Rowling,Fantasy,1997,223,good,"magical,cosy",,1,2026-09-14
3,9780156012195,The Little Prince,Antoine de Saint-Exupery,Fiction,1943,96,good,"cosy,inspiring",,1,2026-09-14
14,9781250301697,The Midnight Library,Matt Haig,Fiction,2020,288,mint,"emotional,cosy",,1,2026-09-14
2,9780143127741,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,1994,607,mint,"mysterious,cosy",,1,2026-09-14
5,9780307949486,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,1994,607,mint,"mysterious,cosy",,1,2026-09-14


In [26]:
display_friends(read_friends())

,Friend id,Friend name,Phone number,Email,Max loans,Trust score,Preferred genres,Date added,Notes
0,1,Emma Watson,07911123456,emma@email.com,3,100,"Fiction,Fantasy",2026-09-14,"Very reliable, always returns on time"
4,5,Olivia Jones,07955567890,olivia@email.com,3,100,"Self-Help,Fiction",2026-09-14,"Best borrower, never late"
5,6,Liane herself,07999000111,liane@library.com,5,100,"Fiction,Fantasy,Romance",2026-09-14,Owner of the library
6,16,Ashritha,07811111111,ashritha@email.com,3,100,"Romance,Fiction",2026-09-14,Very trustworthy
8,18,Zara Ahmed,07833333333,zara@email.com,3,100,"Self-Help,Fiction",2026-09-14,Always on time
9,19,Priya Sharma,07700111222,priya.sharma@email.com,3,100,"Fiction,Romance",2026-09-14,Very reliable
11,21,Mei Lin,07700555666,mei.lin@email.com,3,100,"Self-Help,Fiction",2026-09-14,Always careful with books
2,3,Sophie Turner,07933345678,sophie@email.com,3,90,"Romance,Fiction",2026-09-14,Sometimes a little late
7,17,Noah Bennett,07822222222,noah@email.com,2,90,"Dystopian,Classic",2026-09-14,Sometimes slow to return
10,20,Carlos Rivera,07700333444,carlos.rivera@email.com,2,90,"Fiction,Classic",2026-09-14,Returns on time


In [27]:
display_friends(read_friends(min_trust=80))

,Friend id,Friend name,Phone number,Email,Max loans,Trust score,Preferred genres,Date added,Notes
0,1,Emma Watson,07911123456,emma@email.com,3,100,"Fiction,Fantasy",2026-09-14,"Very reliable, always returns on time"
4,5,Olivia Jones,07955567890,olivia@email.com,3,100,"Self-Help,Fiction",2026-09-14,"Best borrower, never late"
5,6,Liane herself,07999000111,liane@library.com,5,100,"Fiction,Fantasy,Romance",2026-09-14,Owner of the library
6,16,Ashritha,07811111111,ashritha@email.com,3,100,"Romance,Fiction",2026-09-14,Very trustworthy
8,18,Zara Ahmed,07833333333,zara@email.com,3,100,"Self-Help,Fiction",2026-09-14,Always on time
9,19,Priya Sharma,07700111222,priya.sharma@email.com,3,100,"Fiction,Romance",2026-09-14,Very reliable
11,21,Mei Lin,07700555666,mei.lin@email.com,3,100,"Self-Help,Fiction",2026-09-14,Always careful with books
2,3,Sophie Turner,07933345678,sophie@email.com,3,90,"Romance,Fiction",2026-09-14,Sometimes a little late
7,17,Noah Bennett,07822222222,noah@email.com,2,90,"Dystopian,Classic",2026-09-14,Sometimes slow to return
10,20,Carlos Rivera,07700333444,carlos.rivera@email.com,2,90,"Fiction,Classic",2026-09-14,Returns on time


In [28]:
display_loans()

,tracker_id,friend_name,book_name,loan_date,due_date,renewal_date,status,fine_amount,fine_status
0,LIB-000001,Emma Watson,To Kill a Mockingbird,2024-08-01,2024-08-15,2024-09-30,OVERDUE,0.0,none
1,LIB-000002,James Brown,The Great Gatsby,2024-08-05,2024-08-19,None,Returned,378.0,paid
2,LIB-000003,Sophie Turner,Sapiens,2024-08-10,2024-08-24,2024-08-31,OVERDUE,0.0,none
3,LIB-000004,Liam Smith,Harry Potter and the Philosophers Stone,2024-08-12,2024-08-26,None,OVERDUE,0.0,none
4,LIB-000005,Olivia Jones,It Ends with Us,2024-08-15,2024-08-29,2024-09-05,OVERDUE,0.0,none
5,LIB-000006,Emma Watson,Atomic Habits,2024-08-20,2024-09-03,None,OVERDUE,0.0,none
6,LIB-000007,James Brown,The Handmaids Tale,2024-08-22,2024-09-05,None,OVERDUE,0.0,none
7,LIB-000008,Sophie Turner,Crime and Punishment,2024-08-25,2024-09-08,2024-09-15,OVERDUE,0.0,none
8,LIB-000009,Liam Smith,The Midnight Library,2024-09-01,2024-09-15,None,OVERDUE,0.0,none
9,LIB-000010,Olivia Jones,The Subtle Art of Not Giving a F,2024-09-05,2024-09-19,2024-09-26,OVERDUE,0.0,none


In [29]:
# Check the tracker ID of the first new loan we just created
new_tracker = pd.read_sql("SELECT tracker_id FROM loans ORDER BY loan_id DESC LIMIT 3", con=connection_string).iloc[2]['tracker_id']
check_tracker(new_tracker)

,tracker_id,friend_name,book_name,loan_date,due_date,renewal_date,return_date,fine_amount,fine_status
0,LIB-000016,Priya Sharma,The Wind-Up Bird Chronicle,2026-09-14,2026-09-28,2026-10-05,None,0.0,none


In [30]:
read_overdue()

,tracker_id,friend_name,phone_number,book_name,deadline,days_overdue,fine_due
0,LIB-000004,Liam Smith,07944456789,Harry Potter and the Philosophers Stone,2024-08-26,749,374.5
1,LIB-000003,Sophie Turner,07933345678,Sapiens,2024-08-31,744,372.0
2,LIB-000006,Emma Watson,07911123456,Atomic Habits,2024-09-03,741,370.5
3,LIB-000005,Olivia Jones,07955567890,It Ends with Us,2024-09-05,739,369.5
4,LIB-000007,James Brown,07922234567,The Handmaids Tale,2024-09-05,739,369.5
5,LIB-000008,Sophie Turner,07933345678,Crime and Punishment,2024-09-15,729,364.5
6,LIB-000009,Liam Smith,07944456789,The Midnight Library,2024-09-15,729,364.5
7,LIB-000010,Olivia Jones,07955567890,The Subtle Art of Not Giving a F,2024-09-26,718,359.0
8,LIB-000001,Emma Watson,07911123456,To Kill a Mockingbird,2024-09-30,714,357.0


In [31]:
read_reading_progress()

,friend_name,book_name,total_pages,pages_read_so_far,percent_complete,avg_mood
0,Emma Watson,To Kill a Mockingbird,281,205.0,73.0,5.0
1,Carlos Rivera,The Little Prince,96,50.0,52.1,5.0
2,Ashritha,The Alchemist,208,100.0,48.1,5.0
3,Liam Smith,The Midnight Library,288,90.0,31.3,5.0
4,Mei Lin,Watership Down,413,120.0,29.1,5.0
5,Olivia Jones,It Ends with Us,384,100.0,26.0,5.0
6,Liam Smith,Harry Potter and the Philosophers Stone,223,40.0,17.9,3.0
7,James Brown,The Handmaids Tale,311,55.0,17.7,4.0
8,James Brown,The Great Gatsby,180,30.0,16.7,3.0
9,Emma Watson,Atomic Habits,320,45.0,14.1,4.0


In [32]:
library_summary()

,total_books,available,total_friends,active_loans,overdue_loans,unpaid_fines,wishlist_pending
0,17,17,12,15,9,0.0,9


---
# UPDATE ✏️
Functions to modify existing records.

## Define

In [33]:
def format_field(field):
    """Makes field names human readable for messages."""
    special = {
        "isbn": "ISBN", "trust_score": "Trust score",
        "fine_status": "Fine status", "is_available": "Availability"
    }
    return special.get(field, field.replace("_", " ").capitalize())

In [34]:
def update_friend(friend, field, new_value):
    """Updates any field on a friend record."""
    query = f"""
        UPDATE friends
        SET {field} = '{new_value}'
        WHERE friend_id = {friend['friend_id']};
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"✅ {format_field(field)} updated for '{friend['friend_name']}' → '{new_value}'"

In [35]:
def update_book(book, field, new_value):
    """Updates any field on a book record."""
    query = f"""
        UPDATE books
        SET {field} = '{new_value}'
        WHERE isbn = '{book['isbn']}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"✅ {format_field(field)} updated for '{book['book_name']}' → '{new_value}'"

In [36]:
def return_book(tracker_id, return_condition='good'):
    """
    Marks a book as returned.
    The calculate_fine trigger fires automatically.
    return_condition: mint, good, worn, damaged
    """
    query = f"""
        UPDATE loans
        SET return_date      = CURDATE(),
            return_condition = '{return_condition}'
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"📬 Loan {tracker_id} returned in '{return_condition}' condition. Fine calculated automatically."

In [37]:
def renew_loan(tracker_id, extra_days=7):
    """Extends a loan by adding extra days to the renewal date."""
    query = f"""
        UPDATE loans
        SET renewal_date = DATE_ADD(COALESCE(renewal_date, due_date), INTERVAL {extra_days} DAY)
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🔄 Loan {tracker_id} renewed by {extra_days} days."

In [38]:
def pay_fine(tracker_id):
    """Marks a fine as paid for a given tracker ID."""
    query = f"""
        UPDATE loans
        SET fine_status = 'paid'
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"💰 Fine marked as PAID for loan {tracker_id}."

In [39]:
def lower_trust_score(friend, points=10, reason=None):
    """Lowers a friend's trust score. Will not go below 0."""
    query = f"""
        UPDATE friends
        SET trust_score = GREATEST(0, trust_score - {points})
        WHERE friend_id = {friend['friend_id']};
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    msg = f"⚠️ Trust score for '{friend['friend_name']}' reduced by {points} points."
    if reason:
        msg += f" Reason: {reason}"
    return msg

In [40]:
def fulfill_wishlist(wishlist_id):
    """Marks a wishlist request as fulfilled when Liane gets the book."""
    query = f"UPDATE wishlist SET fulfilled = TRUE WHERE wishlist_id = {wishlist_id};"
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🎁 Wishlist item {wishlist_id} marked as fulfilled!"

## Test UPDATE

In [41]:
# Update Carlos's details
carlos = pd.read_sql("SELECT * FROM friends WHERE email = 'carlos.rivera@email.com'", con=connection_string).iloc[0]

print(update_friend(carlos, 'max_loans', 3))
print(update_friend(carlos, 'notes', 'Updated - extremely reliable borrower'))

pd.read_sql("SELECT * FROM friends WHERE email = 'carlos.rivera@email.com'", con=connection_string)

✅ Max loans updated for 'Carlos Rivera' → '3'
✅ Notes updated for 'Carlos Rivera' → 'Updated - extremely reliable borrower'


,friend_id,friend_name,phone_number,email,max_loans,trust_score,preferred_genres,date_added,notes
0,20,Carlos Rivera,07700333444,carlos.rivera@email.com,3,90,"Fiction,Classic",2026-09-14,Updated - extremely reliable borrower


In [42]:
# Update Watership Down's condition
book = pd.read_sql("SELECT * FROM books WHERE isbn = '9780385333481'", con=connection_string).iloc[0]

print(update_book(book, 'book_condition', 'good'))
print(update_book(book, 'mood_tags', 'emotional,inspiring,cosy'))

pd.read_sql("SELECT * FROM books WHERE isbn = '9780385333481'", con=connection_string)

✅ Book condition updated for 'Watership Down' → 'good'
✅ Mood tags updated for 'Watership Down' → 'emotional,inspiring,cosy'


,isbn,book_name,author,genre,published_year,total_pages,book_condition,mood_tags,cover_url,is_available,date_added
0,9780385333481,Watership Down,Richard Adams,Fiction,1972,413,good,"emotional,inspiring,cosy",None,1,2026-09-14


In [43]:
# Renew Priya's loan
priya_loan = pd.read_sql("""
    SELECT l.tracker_id FROM loans l
    JOIN friends f ON l.friend_id = f.friend_id
    WHERE f.email = 'priya.sharma@email.com' AND l.return_date IS NULL
""", con=connection_string)

tracker = priya_loan.iloc[0]['tracker_id']
print(renew_loan(tracker, extra_days=7))
check_tracker(tracker)

🔄 Loan LIB-000016 renewed by 7 days.


,tracker_id,friend_name,book_name,loan_date,due_date,renewal_date,return_date,fine_amount,fine_status
0,LIB-000016,Priya Sharma,The Wind-Up Bird Chronicle,2026-09-14,2026-09-28,2026-10-12,None,0.0,none


In [44]:
# Return Carlos's book — fine calculated automatically by trigger
carlos_loan = pd.read_sql("""
    SELECT l.tracker_id FROM loans l
    JOIN friends f ON l.friend_id = f.friend_id
    WHERE f.email = 'carlos.rivera@email.com' AND l.return_date IS NULL
""", con=connection_string)

tracker = carlos_loan.iloc[0]['tracker_id']
print(return_book(tracker, return_condition='good'))
check_tracker(tracker)

📬 Loan LIB-000017 returned in 'good' condition. Fine calculated automatically.


,tracker_id,friend_name,book_name,loan_date,due_date,renewal_date,return_date,fine_amount,fine_status
0,LIB-000017,Carlos Rivera,The Little Prince,2026-09-14,2026-10-05,2026-10-12,2026-09-14,0.0,none


In [45]:
# Pay Carlos's fine
print(pay_fine(tracker))
check_tracker(tracker)

💰 Fine marked as PAID for loan LIB-000017.


,tracker_id,friend_name,book_name,loan_date,due_date,renewal_date,return_date,fine_amount,fine_status
0,LIB-000017,Carlos Rivera,The Little Prince,2026-09-14,2026-10-05,2026-10-12,2026-09-14,0.0,paid


In [46]:
# Lower Mei's trust score — returned book with a torn page
mei = pd.read_sql("SELECT * FROM friends WHERE email = 'mei.lin@email.com'", con=connection_string).iloc[0]
print(lower_trust_score(mei, points=10, reason="Returned book with torn page"))

pd.read_sql("SELECT friend_name, trust_score FROM friends ORDER BY trust_score DESC", con=connection_string)

⚠️ Trust score for 'Mei Lin' reduced by 10 points. Reason: Returned book with torn page


,friend_name,trust_score
0,Emma Watson,100
1,Olivia Jones,100
2,Liane herself,100
3,Ashritha,100
4,Zara Ahmed,100
5,Priya Sharma,100
6,Sophie Turner,90
7,Noah Bennett,90
8,Carlos Rivera,90
9,Mei Lin,90


In [47]:
# Mark Priya's wishlist item as fulfilled
priya_wish = pd.read_sql("""
    SELECT w.wishlist_id FROM wishlist w
    JOIN friends f ON w.friend_id = f.friend_id
    WHERE f.email = 'priya.sharma@email.com' AND w.fulfilled = 0
""", con=connection_string)

print(fulfill_wishlist(priya_wish.iloc[0]['wishlist_id']))
pd.read_sql("wishlist", con=connection_string)

🎁 Wishlist item 8 marked as fulfilled!


,wishlist_id,friend_id,book_title,author,requested_on,fulfilled
0,1,1,The Alchemist,Paulo Coelho,2024-08-10,1
1,2,2,Thinking Fast and Slow,Daniel Kahneman,2024-08-12,0
2,3,3,Pride and Prejudice,Jane Austen,2024-08-15,0
3,4,4,1984,George Orwell,2024-08-18,0
4,5,5,The Power of Now,Eckhart Tolle,2024-08-20,0
5,6,18,The Power of Now,Eckhart Tolle,2026-09-14,0
6,7,17,1984,George Orwell,2026-09-14,0
7,8,19,Norwegian Wood,Haruki Murakami,2026-09-14,1
8,9,20,One Hundred Years of Solitude,Gabriel Garcia Marquez,2026-09-14,0
9,10,21,Educated,Tara Westover,2026-09-14,0


---
# DELETE 🗑️
Functions to remove records safely.

## Define

In [63]:
def delete_friend(friend):
    """
    Removes a friend and ALL their linked data in the correct order.
    Order: reading_sessions → reviews → loans → wishlist → friends
    """
    fid = friend['friend_id']

    with engine.begin() as conn:
        # 1. Get all loan IDs for this friend
        loans = pd.read_sql(
            f"SELECT loan_id FROM loans WHERE friend_id = {fid}",
            con=connection_string
        )

        # 2. Delete reading_sessions and reviews for each loan
        for loan_id in loans['loan_id']:
            conn.execute(text(f"DELETE FROM reading_sessions WHERE loan_id = {loan_id};"))
            conn.execute(text(f"DELETE FROM reviews          WHERE loan_id = {loan_id};"))

        # 3. Delete all loans
        conn.execute(text(f"DELETE FROM loans    WHERE friend_id = {fid};"))

        # 4. Delete wishlist entries
        conn.execute(text(f"DELETE FROM wishlist WHERE friend_id = {fid};"))

        # 5. Finally delete the friend
        conn.execute(text(f"DELETE FROM friends  WHERE friend_id = {fid};"))

    return f"🗑️ '{friend['friend_name']}' and all their linked data removed successfully."

In [64]:
def delete_book(book):
    """Removes a book. Blocked if it is currently on loan."""
    active = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE isbn = '{book['isbn']}' AND return_date IS NULL",
        con=connection_string
    )
    if not active.empty:
        return f"⚠️ '{book['book_name']}' is currently on loan — cannot delete."
    with engine.begin() as conn:
        conn.execute(text(f"DELETE FROM books WHERE isbn = '{book['isbn']}';"))
    return f"🗑️ '{book['book_name']}' removed from the library."

In [65]:
def delete_loan(tracker_id):
    """Removes a loan and its linked reading sessions and reviews."""
    loan = pd.read_sql(
        f"SELECT * FROM loans WHERE tracker_id = '{tracker_id}'",
        con=connection_string
    )
    if loan.empty:
        return f"⚠️ No loan found with tracker ID '{tracker_id}'."
    loan_id = int(loan.iloc[0]["loan_id"])
    with engine.begin() as conn:
        conn.execute(text(f"DELETE FROM reading_sessions WHERE loan_id = {loan_id};"))
        conn.execute(text(f"DELETE FROM reviews          WHERE loan_id = {loan_id};"))
        conn.execute(text(f"DELETE FROM loans            WHERE loan_id = {loan_id};"))
    return f"🗑️ Loan {tracker_id} and its linked sessions/reviews deleted."

In [66]:
def delete_wishlist(wishlist_id):
    """Removes a wishlist request."""
    with engine.begin() as conn:
        conn.execute(text(f"DELETE FROM wishlist WHERE wishlist_id = {wishlist_id};"))
    return f"🗑️ Wishlist item {wishlist_id} removed."

## Test DELETE

In [67]:
# Check friends before deleting
table_pre = pd.read_sql("friends", con=connection_string)
table_pre

,friend_id,friend_name,phone_number,email,max_loans,trust_score,preferred_genres,date_added,notes
0,1,Emma Watson,07911123456,emma@email.com,3,100,"Fiction,Fantasy",2026-09-14,"Very reliable, always returns on time"
1,2,James Brown,07922234567,james@email.com,2,55,"Non-Fiction,Self-Help",2026-09-14,Returned one book damaged
2,3,Sophie Turner,07933345678,sophie@email.com,3,90,"Romance,Fiction",2026-09-14,Sometimes a little late
3,4,Liam Smith,07944456789,liam@email.com,2,60,"Classic,Dystopian",2026-09-14,"Lost a book once, be careful"
4,5,Olivia Jones,07955567890,olivia@email.com,3,100,"Self-Help,Fiction",2026-09-14,"Best borrower, never late"
5,6,Liane herself,07999000111,liane@library.com,5,100,"Fiction,Fantasy,Romance",2026-09-14,Owner of the library
6,16,Ashritha,07811111111,ashritha@email.com,3,100,"Romance,Fiction",2026-09-14,Very trustworthy
7,17,Noah Bennett,07822222222,noah@email.com,2,90,"Dystopian,Classic",2026-09-14,Sometimes slow to return
8,18,Zara Ahmed,07833333333,zara@email.com,3,100,"Self-Help,Fiction",2026-09-14,Always on time
9,19,Priya Sharma,07700111222,priya.sharma@email.com,3,100,"Fiction,Romance",2026-09-14,Very reliable


In [68]:
carlos = pd.read_sql("SELECT * FROM friends WHERE email = 'carlos.rivera@email.com'", con=connection_string).iloc[0]
print(delete_friend(carlos))

🗑️ 'Carlos Rivera' and all their linked data removed successfully.


In [69]:
# Confirm row is gone — compare before and after
table_post = pd.read_sql("friends", con=connection_string)
dropped = pd.concat([table_pre, table_post]).drop_duplicates(keep=False)
print("Removed row:")
dropped

Removed row:


,friend_id,friend_name,phone_number,email,max_loans,trust_score,preferred_genres,date_added,notes
10,20,Carlos Rivera,07700333444,carlos.rivera@email.com,3,90,"Fiction,Classic",2026-09-14,Updated - extremely reliable borrower


In [70]:
# Delete Mei's loan safely (removes linked sessions too)
mei_loan = pd.read_sql("""
    SELECT l.tracker_id FROM loans l
    JOIN friends f ON l.friend_id = f.friend_id
    WHERE f.email = 'mei.lin@email.com'
    ORDER BY l.loan_id DESC LIMIT 1
""", con=connection_string)

loans_pre = pd.read_sql("loans", con=connection_string)
tracker   = mei_loan.iloc[0]['tracker_id']
print(delete_loan(tracker))

loans_post  = pd.read_sql("loans", con=connection_string)
dropped_loan = pd.concat([loans_pre, loans_post]).drop_duplicates(keep=False)
print("\nRemoved loan:")
print(dropped_loan[['loan_id', 'tracker_id', 'isbn', 'friend_id']].to_string(index=False))

🗑️ Loan LIB-000018 and its linked sessions/reviews deleted.

Removed loan:
 loan_id tracker_id          isbn  friend_id
      18 LIB-000018 9780385333481         21


---
# VALIDATE ✅
Validation functions that check data BEFORE it goes into the database.

## Define

In [71]:
def validate_name(name):
    """Checks a name is not empty."""
    if not name or not str(name).strip():
        return "⚠️ Name cannot be empty."
    return ""

In [72]:
def validate_isbn(isbn):
    """Checks ISBN is 10 or 13 digits and not already in the database."""
    isbn = str(isbn)
    if len(isbn) not in (10, 13):
        return "⚠️ ISBN must be 10 or 13 digits long."
    if not isbn.isnumeric():
        return "⚠️ ISBN must contain numbers only."
    existing = pd.read_sql("SELECT isbn FROM books", con=connection_string)["isbn"].values
    if isbn in existing:
        return "⚠️ This ISBN already exists in the library."
    return ""

In [73]:
def validate_email(email):
    """Checks email format and that it is not already registered."""
    if not email or "@" not in str(email) or "." not in str(email):
        return "⚠️ Please enter a valid email address."
    existing = pd.read_sql("SELECT email FROM friends", con=connection_string)["email"].values
    if email in existing:
        return "⚠️ This email is already registered."
    return ""

In [74]:
def validate_loan_friend(friend):
    """Checks friend can borrow. Blocks at max loans. Warns if trust is low."""
    active_loans = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE friend_id = {friend['friend_id']} AND return_date IS NULL",
        con=connection_string
    )
    if len(active_loans) >= friend["max_loans"]:
        return f"⚠️ '{friend['friend_name']}' has reached their max loan limit ({friend['max_loans']})."
    if friend["trust_score"] < 50:
        return f"⚠️ '{friend['friend_name']}' has a low trust score ({friend['trust_score']}). Lend with caution!"
    return ""

In [75]:
def validate_loan_book(book):
    """Checks a book is available and not damaged."""
    on_loan = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE isbn = '{book['isbn']}' AND return_date IS NULL",
        con=connection_string
    )
    if not on_loan.empty:
        return f"⚠️ '{book['book_name']}' is already out on loan."
    if book["book_condition"] == "damaged":
        return f"⚠️ '{book['book_name']}' is marked as damaged. Should it be lent out?"
    return ""

In [76]:
def validate_rating(rating):
    """Checks rating is a whole number between 1 and 5."""
    if not isinstance(rating, int) or rating < 1 or rating > 5:
        return "⚠️ Rating must be a whole number between 1 and 5."
    return ""

## Test VALIDATE

In [77]:
# Name validation
print(validate_name("Priya"))  # ✅ pass
print(validate_name(""))        # ⚠️ empty
print(validate_name("   "))     # ⚠️ spaces
print(validate_name(None))      # ⚠️ None


⚠️ Name cannot be empty.
⚠️ Name cannot be empty.
⚠️ Name cannot be empty.


In [78]:
# ISBN validation
print(validate_isbn("9780307949486"))  # ⚠️ already in DB
print(validate_isbn("123"))             # ⚠️ too short
print(validate_isbn("97803079494ab"))   # ⚠️ not numeric
print(validate_isbn("9998887776665"))   # ✅ valid new ISBN

⚠️ This ISBN already exists in the library.
⚠️ ISBN must be 10 or 13 digits long.
⚠️ ISBN must contain numbers only.



In [79]:
# Email validation
print(validate_email("priya.sharma@email.com"))  # ⚠️ already registered
print(validate_email("notanemail"))               # ⚠️ invalid format
print(validate_email("brand.new@email.com"))      # ✅ valid

⚠️ This email is already registered.
⚠️ Please enter a valid email address.



In [80]:
# Loan friend validation — use Priya and Mei who we added above
priya = pd.read_sql("SELECT * FROM friends WHERE email = 'priya.sharma@email.com'", con=connection_string).iloc[0]
mei   = pd.read_sql("SELECT * FROM friends WHERE email = 'mei.lin@email.com'",      con=connection_string).iloc[0]

print(validate_loan_friend(priya))  # ✅ should pass
print(validate_loan_friend(mei))    # may warn if trust score dropped

In [81]:
# Loan book validation — use our new books
book_free   = pd.read_sql("SELECT * FROM books WHERE isbn = '9780156012195'", con=connection_string).iloc[0]  # The Little Prince
book_on_loan = pd.read_sql("SELECT * FROM books WHERE isbn = '9780307949486'", con=connection_string).iloc[0] # Wind-Up Bird

print(validate_loan_book(book_free))    # ✅ available
print(validate_loan_book(book_on_loan)) # ⚠️ on loan


⚠️ 'The Wind-Up Bird Chronicle' is already out on loan.


In [82]:
# Rating validation
print(validate_rating(5))    # ✅ pass
print(validate_rating(0))    # ⚠️ too low
print(validate_rating(6))    # ⚠️ too high
print(validate_rating(3.5))  # ⚠️ not integer


⚠️ Rating must be a whole number between 1 and 5.
⚠️ Rating must be a whole number between 1 and 5.
⚠️ Rating must be a whole number between 1 and 5.


---
# All functions are ready ✅

These CRUD functions can now be imported directly into the Streamlit app:

```python
from crud import (
    create_book, create_friend, create_loan,
    create_reading_session, create_wishlist_request, create_review,
    read_books, read_friends, display_loans, check_tracker,
    read_overdue, read_reading_progress, library_summary,
    update_friend, update_book, return_book, renew_loan,
    pay_fine, lower_trust_score, fulfill_wishlist,
    delete_friend, delete_book, delete_loan, delete_wishlist,
    validate_name, validate_isbn, validate_email,
    validate_loan_friend, validate_loan_book, validate_rating
)
```